___
# Checking Object Validity
Gary Mitchell
March 18, 2020

---

One of the major benefits of object oriented programming (OOP) is
that individual objects are able to validate their state. An 
object in an invalid state can/should refuse to perform an action
that would be negatively impacted/corrupted by its invalid state.

Consider a Ball for example. A Ball with a negative radius cannot
exist. A negative radius would imply that the Ball had imploded.

There are two ways we could go about creating such a self-aware
Ball class:

-   Never allow the Ball to have an invalid radius
-   Allow an invalid radius, but require every function that
depends on the radius to check the validity.

## The Ball class

Let's look at a basic Ball class:

In [15]:
import math
class Ball:
    def __init__(self, radius):
        self._radius = radius
        
    @property
    def area(self):
        if self.is_valid():
            return math.pi * self._radius ** 2
        else:
            return math.nan     
        
    def is_valid(self):
        if self._radius > 0:
            return True
        else:
            return False      
       

In the Ball class defined above, we can create a Ball with a 
negative radius. Of course, the Ball would immediately implode
(that *could* be fun...). In order to correctly calculate the
surface area, we must first determine whether the Ball instance
is valid. If it is, we can calculate the radius, if it isn't,
then we will return NaN (because the Ball imploded, we'll 
use infinity to indicate that the Ball's area cannot be 
defined.

Here's our Ball in action:

In [16]:
ball1 = Ball(5)
print(ball1.area)

ball2 = Ball(-5)
print(ball2.area)

78.53981633974483
nan


Note that the area property had to explicitly check for validity. What
would have happened if the programmer simply *forget* to do the validity
test? The code below illustrates:

In [17]:
class BadBall:
    def __init__(self, radius):
        self._radius = radius
        
    @property
    def area(self):
        return math.pi * self._radius ** 2
   
        
    def is_valid(self):
        if self._radius > 0:
            return True
        else:
            return False      

BadBall differs from Ball in only one respect: the area property does
not test validity before performing its calculation. Let's look at
the impact.

In [18]:
ball1 = BadBall(5)
print(ball1.area)

ball2 = BadBall(-5)
print(ball2.area)

78.53981633974483
78.53981633974483


Oops! That clearly can't be right. The imploded ball shows as having
a radius of 78.54 (approximate)! This is because BadBall relies on
all of its methods to adjust for its sloppy design failing to prevent
an invalid state.

On the other hand, BetterBall (defined below), solves this problem
by prevening radius from every containing an *invalid* value:

In [19]:
class BetterBall:
    def __init__(self, radius):
        if radius >= 0:
            self._radius = radius
        else:
            self._radius = math.nan
        
    @property
    def area(self):
        return math.pi * self._radius ** 2
   
        
    def is_valid(self):
        if math.isnan(self._radius):
            return False
        else:
            return True      

Here, the area property is the same as BadBall's, but the constructor
has changed. On construction, BetterBall ensures that the radius
contains either a valid value (non-negative) or NaN. Because NaN
in any calculation produces NaN, the area property will return
the correct result, *regardless of whether it first checks for
object validity*. That's huge. Here we see that BetterBall instances
will return correct area results even though they do not check to
see if they are valid.

In [20]:
ball1 = BetterBall(5)
print(ball1.area)

ball2 = BetterBall(-5)
print(ball2.area)

78.53981633974483
nan


An additional advantage of this approach is that if the test
for validity were more involved (e.g. testing validity involved
testing multiple attributes rather than just one), we would limit
the need for testing that to construction. All calls to is_valid
would simply *report* the validity state (True/False) that the
object has already determined, saving the need to repeatedly
evaluate validity *each time is_valid is called*.